**Phase kickback.**
**Phase kickback** is a fundamental quantum phenomenon where applying a controlled
gate to a target qubit that is an *eigenstate* of the gate's unitary causes
the associated eigenvalue phase to be transferred back to the *control* qubit.
It is the core mechanism behind Deutsch's algorithm, the quantum Fourier
transform, and quantum phase estimation.

If a unitary $U$ has eigenstate $|u\rangle$ with eigenvalue $e^{i\phi}$,
then a controlled-$U$ gate acts as:

$$\text{C-}U\,\frac{|0\rangle+|1\rangle}{\sqrt{2}}\otimes|u\rangle
= \frac{|0\rangle + e^{i\phi}|1\rangle}{\sqrt{2}}\otimes|u\rangle$$

The target is unchanged, but the control qubit acquires the phase $e^{i\phi}$.
This notebook builds up to phase kickback through five progressions:

1. **Cells 01-03**: tensor product representations in NumPy
2. **Cells 04-05**: single-qubit circuits showing the $|-\rangle$ state and Rz rotation
3. **Cells 06-07**: CNOT with and without phase kickback
4. **Cells 08-09**: controlled-phase gate with and without kickback
5. **Cells 10-11**: quantum phase estimation (exact and near match)

A single `AerSimulator` backend is shared across all Qiskit cells.

---
**Cell 01 - Tensor product of a 1-D row vector.**
`np.array([0, 1])` creates a 1-D array of shape `(2,)` - NumPy treats this
as a row vector, not a column.
`np.kron` computes the Kronecker product correctly regardless of shape,
but the result inherits the 1-D structure: `T3` has shape `(8,)` and
`as_latex` renders it as a horizontal row.
This represents $|1\rangle^{\otimes 3} = |111\rangle$,
whose only nonzero entry is the last element (index 7).

In [ ]:
"""phase_kickback.ipynb"""

# Cell 01 - Tensor product using a 1-D row vector  (shape: (2,))

import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_bloch_multivector, plot_distribution
from qiskit_aer import AerSimulator

from qis101_utils import as_latex

backend = AerSimulator()


def print_ndarray_info(name, a):
    print(f"Type of {name} is {type(a).__name__}")
    print(f"Number of dimensions of {name} = {a.ndim}")
    print(f"Shape of dimensions of {name} = {a.shape}")
    print(f"Length of {name} = {len(a)}")
    print(f"Size of {name} = {a.size}")


t = np.array([0, 1])  # 1-D row vector, shape (2,)
display(as_latex(t, prefix=r"\mathbf{T}=1="))
print_ndarray_info("T", t)

t3 = np.kron(t, np.kron(t, t))
display(as_latex(t3, prefix=r"\mathbf{T^{\otimes 3}}="))
print_ndarray_info("T3", t3)

---
**Cell 02 - Tensor product of a 2-D column vector.**
Defining `t` as `np.array([[0], [1]])` gives shape `(2, 1)` - a true 2-D column.
`np.kron` on two `(2, 1)` arrays produces a `(8, 1)` result, which
`as_latex` renders as a vertical column vector.
The mathematical content is identical to Cell 01 - the difference is
purely representational and affects how `as_latex` and downstream
matrix operations interpret the array.

In [ ]:
# Cell 02 - Tensor product using a 2-D column vector (shape: (2,1))
# as_latex renders this as a column rather than a row

t = np.array([[0], [1]])  # 2-D column vector, shape (2,1)
display(as_latex(t, prefix=r"\mathbf{T}=1="))
print_ndarray_info("T", t)

t3 = np.kron(t, np.kron(t, t))
display(as_latex(t3, prefix=r"\mathbf{T^{\otimes 3}}="))
print_ndarray_info("T3", t3)

---
**Cell 03 - Promoting a 1-D array to a column vector.**
Starting again with a 1-D `t` of shape `(2,)`, this cell shows two ways
to display `T3` as a column without redefining `t` as 2-D:
`as_latex(..., column=True)` renders it vertically in place,
and `t3[:, np.newaxis]` permanently promotes the `(8,)` array
to a `(8, 1)` 2-D column by inserting a new axis.
The three `as_latex` calls produce identical LaTeX output;
the `print_ndarray_info` calls show the shape change.

In [ ]:
# Cell 03 - Promote a 1-D array to a column vector using [:, np.newaxis]
# column=True in as_latex renders a 1-D array vertically without promotion

t = np.array([0, 1])
print_ndarray_info("T", t)
display(as_latex(t, prefix=r"\mathbf{T}=1="))

t3 = np.kron(t, np.kron(t, t))
print_ndarray_info("T3", t3)
display(as_latex(t3, prefix=r"\mathbf{T^{\otimes 3}}="))
display(as_latex(t3, prefix=r"\mathbf{T^{\otimes 3}}=", column=True))

# [:, np.newaxis] promotes to true (8,1) column - same result, different shape
t3 = t3[:, np.newaxis]
print_ndarray_info("T3", t3)
display(as_latex(t3, prefix=r"\mathbf{T^{\otimes 3}}="))

---
**Cell 04 - X then H: preparing the $|-\rangle$ state.**
Starting from $|0\rangle$: X flips to $|1\rangle$, then H maps $|1\rangle \to |-\rangle$:

$$|0\rangle \xrightarrow{X} |1\rangle
\xrightarrow{H} |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

$|-\rangle$ is an eigenstate of the X (Pauli-X) gate with eigenvalue $-1$.
This is the target state used in phase kickback with CNOT:
when the target qubit is in $|-\rangle$, any CNOT kicks back a $-1$ phase
to the control qubit.
Measurement in the Z basis gives 50/50 since $|-\rangle$ has equal amplitudes.

In [ ]:
# Cell 04 - X then H: |0> -> |1> -> |-> = (|0>-|1>)/sqrt2
# |--> is the -1 eigenstate of X, key for phase kickback

qc = QuantumCircuit(1)
qc.x(0)
qc.save_statevector("sv1")
qc.h(0)
qc.save_statevector("sv2")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 05 - $R_y(\pi/2)$ then $R_z(\pi)$: rotation to $|-\rangle$ via different gates.**
$R_y(\pi/2)$ rotates from $|0\rangle$ to $|{+}\rangle$ on the Bloch sphere;
$R_z(\pi)$ then applies a $\pi$ phase to the $|1\rangle$ component,
rotating from $|{+}\rangle$ to $|-\rangle$ (up to global phase).
This demonstrates an alternative path to the same final state.

In [ ]:
# Cell 05 - Ry(pi/2) then Rz(pi): alternative route to |-> (up to global phase)

qc = QuantumCircuit(1)
qc.ry(np.pi / 2, 0)
qc.save_statevector("sv1")
qc.rz(np.pi, 0)
qc.save_statevector("sv2")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 06 - CNOT with $|{+}\rangle|{+}\rangle$ input: no phase kickback.**
Both qubits start in $|0\rangle$ and receive Hadamard gates, giving
$|{+}\rangle|{+}\rangle$.
The CNOT's target q1 is in $|{+}\rangle$, which is the $+1$ eigenstate of X:
$X|{+}\rangle = +1\cdot|{+}\rangle$.
A kickback phase of $+1$ is physically undetectable - multiplying by 1
leaves the control qubit unchanged.
The trailing two Hadamard gates acting on both qubits map back to $|00\rangle$,
so measurement is deterministic (always `00`).

In [ ]:
# Cell 06 - CNOT without phase kickback: target q1 in |+> (+1 eigenstate of X)
# H|+>H|+> -> CNOT -> H H -> |00> deterministic

qc = QuantumCircuit(2)
qc.save_statevector("sv1")
qc.h(0)
qc.h(1)
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.save_statevector("sv3")
qc.h(0)
qc.h(1)
qc.save_statevector("sv4")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))
display(plot_bloch_multivector(sv4))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 07 - CNOT with $|{+}\rangle|-\rangle$ input: now phase kickback occurs.**
X on q1 followed by H puts q1 in $|-\rangle = (|0\rangle-|1\rangle)/\sqrt{2}$,
the $-1$ eigenstate of X.
When the CNOT fires (q0=$|1\rangle$), it multiplies the $|-\rangle$ state
by its eigenvalue $-1$, which kicks back to the *control*:

$$\text{CNOT}\,|{+}\rangle|-\rangle
= \frac{1}{\sqrt{2}}(|0\rangle|-\rangle + |1\rangle X|-\rangle)
= \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)|-\rangle
= |-\rangle|-\rangle$$

The control q0 has flipped from $|{+}\rangle$ to $|-\rangle$.
The final H on both qubits then maps q0 back to $|1\rangle$ (not $|0\rangle$),
so measurement gives `11` with probability 1.

In [ ]:
# Cell 07 - CNOT WITH phase kickback: target q1 in |-> (-1 eigenstate of X)
# X(q1) -> H(q0),H(q1) -> CNOT -> H(q0),H(q1) -> |11> deterministic
# The -1 eigenvalue kicks back: control q0 flips from |+> to |->

qc = QuantumCircuit(2)
qc.x(1)  # q1: |0> -> |1>
qc.save_statevector("sv1")
qc.h(0)  # q0: |0> -> |+>
qc.h(1)  # q1: |1> -> |->
qc.save_statevector("sv2")
qc.cx(0, 1)  # phase -1 kicks back to control: q0 -> |->
qc.save_statevector("sv3")
qc.h(0)  # q0: |-> -> |1>
qc.h(1)  # q1: |-> -> |1>
qc.save_statevector("sv4")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))
display(plot_bloch_multivector(sv4))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 08 - Controlled-phase gate with target in $|0\rangle$: no kickback.**
The controlled-phase gate $\text{CP}(\theta)$ applies $P(\theta) = \begin{bmatrix}1&0\\0&e^{i\theta}\end{bmatrix}$
to the target when the control is $|1\rangle$.
Here q1 starts in $|0\rangle$, which is an eigenstate of $P(\theta)$ with eigenvalue 1.
A phase of 1 carries no information, so no phase kicks back to q0.
The measurement of q0 after H gives 50/50.

In [ ]:
# Cell 08 - CP(pi/4) without phase kickback: target q1 in |0> (trivial eigenvalue 1)
# q0: H -> |+>, CP -> no kickback, H -> |+> measured 50/50

qc = QuantumCircuit(2, 1)
qc.h(0)
qc.save_statevector("sv1")
qc.cp(np.pi / 4, 0, 1)  # controlled-P(pi/4): no kickback since q1 in |0>
qc.save_statevector("sv2")
qc.h(0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 09 - Controlled-phase gate with target in $|1\rangle$: kickback occurs.**
$|1\rangle$ is an eigenstate of $P(\theta)$ with eigenvalue $e^{i\theta}$:
$P(\theta)|1\rangle = e^{i\theta}|1\rangle$.
When the control q0=$|1\rangle$ fires, this eigenvalue kicks back:

$$\text{CP}(\theta)\,\frac{|0\rangle+|1\rangle}{\sqrt{2}}\otimes|1\rangle
= \frac{|0\rangle + e^{i\theta}|1\rangle}{\sqrt{2}}\otimes|1\rangle$$

The phase $e^{i\pi/4}$ is now encoded in the control qubit.
After H on q0 the Bloch vector has rotated - measurement is no longer 50/50.

In [ ]:
# Cell 09 - CP(pi/4) WITH phase kickback: target q1 in |1> (eigenvalue e^{i*pi/4})
# q0 acquires phase: (|0> + e^{i*pi/4}|1>)/sqrt2
# After H: measurement is no longer 50/50

qc = QuantumCircuit(2, 1)
qc.h(0)  # q0: |+>
qc.x(1)  # q1: |1> (eigenstate of P with eigenvalue e^{i*pi/4})
qc.save_statevector("sv1")
qc.cp(np.pi / 4, 0, 1)  # phase e^{i*pi/4} kicks back to q0
qc.save_statevector("sv2")
qc.h(0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend), shots=100_000).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 10 - Quantum phase estimation: exact match.**
Phase estimation asks: given a unitary $U$ with unknown eigenvalue $e^{i\phi}$,
determine $\phi$.
Here we know $\phi = \pi/4$ from $\text{CP}(\pi/4)$.
We pre-rotate q0 by $P(-\pi/4)$ to "cancel" the expected kickback:

$$q_0 \text{ after P}(-\pi/4)\text{ and CP}(\pi/4) \text{ kickback}:
\frac{|0\rangle + e^{-i\pi/4}\cdot e^{i\pi/4}|1\rangle}{\sqrt{2}}
= \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |{+}\rangle$$

The final H maps $|{+}\rangle \to |0\rangle$, so measurement gives 0 with
probability 1 - confirming the phase estimate is exact.

In [ ]:
# Cell 10 - Phase estimation (exact match): P(-pi/4) cancels CP(pi/4) kickback
# q0 returns to |+>, H maps |+> -> |0>, measurement always 0

qc = QuantumCircuit(2, 1)
qc.h(0)
qc.p(-np.pi / 4, 0)  # pre-rotate by the negative of the expected phase
qc.x(1)  # q1: |1> (eigenstate)
qc.save_statevector("sv1")
qc.cp(np.pi / 4, 0, 1)  # kickback e^{i*pi/4} cancels the P(-pi/4)
qc.save_statevector("sv2")
qc.h(0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend), shots=100_000).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 11 - Quantum phase estimation: near match.**
If the pre-rotation is $P(-\pi/8)$ instead of $P(-\pi/4)$, the cancellation
is incomplete:

$$\frac{|0\rangle + e^{-i\pi/8}\cdot e^{i\pi/4}|1\rangle}{\sqrt{2}}
= \frac{|0\rangle + e^{i\pi/8}|1\rangle}{\sqrt{2}}$$

After H, q0 is not exactly $|0\rangle$, so measurement gives 0 with probability
$\cos^2(\pi/16) \approx 96\%$ and 1 with probability $\sin^2(\pi/16) \approx 4\%$.
The closer the trial phase is to the true phase, the higher the probability
of measuring 0 - this is how Quantum Phase Estimation narrows down an unknown phase.

In [ ]:
# Cell 11 - Phase estimation (near match): P(-pi/8) vs actual phase pi/4
# Residual phase e^{i*pi/8} remains: P(0 measures) = cos^2(pi/16) ~96%

qc = QuantumCircuit(2, 1)
qc.h(0)
qc.p(-np.pi / 8, 0)  # trial phase: off by pi/8
qc.x(1)
qc.save_statevector("sv1")
qc.cp(np.pi / 4, 0, 1)  # actual phase pi/4 kicks back
qc.save_statevector("sv2")
qc.h(0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend), shots=100_000).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(plot_distribution(result.get_counts(qc)))